### Retrieval-Augmented Generation (RAG) – Single Query Answering System

In [1]:
from langchain_community.vectorstores import Chroma 
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

persistent_directory = "db/chroma_db"

STEP 1: Create document_retriver function 

In [2]:
def document_retriver(persistent_directory,query):

    # Load embeddings and vector store
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    #Load vector database and create vectorstore object
    vectorstore = Chroma(
        persist_directory=persistent_directory,
        embedding_function=embedding_model,
        collection_metadata={"hnsw:space": "cosine"}  
    )
    
    #create retriver object for top 5 results
    retriver = vectorstore.as_retriever(search_kwargs={"k": 5})

    # Retrieve the top-5 most relevant documents for the given query from the vector store
    simillar_documents = retriver.invoke(query)

    return simillar_documents



In [3]:
relevant_docs = document_retriver(persistent_directory,"How much did Microsoft pay to acquire GitHub?")

C:\Users\gihan\AppData\Local\Temp\ipykernel_25420\2791280644.py:7: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [4]:
len(relevant_docs)

5

In [5]:
for i,doc in enumerate (document_retriver(persistent_directory,"How much did Microsoft pay to acquire GitHub?")):
    print(f"\n Document no {i}:")
    print(f"{doc.page_content}")


 Document no 0:
117. "Microsoft's 2018, part 1: Open source, wobbly Windows and everyone's going to the cloud"
(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.
Archived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/
12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved
January 3, 2019.

118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso
ft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or
g/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-
for-7-5-billion/) from the original on June 4, 2018.

 Document no 1:
In April 2018, Microsoft released the source code for Windows File Manager under the MIT License to
celebrate the program's 20th anniversary. In April the company further expressed willingness to embrace
open source initiatives by announcing Azure Sph

In [ ]:
#Combine top 5 simillar documents and create one paragraph
text = ""

for i,doc in enumerate (document_retriver(persistent_directory,"How much did Microsoft pay to acquire GitHub?")):
    text += "document no "+str(i)+" :" + doc.page_content
    text += "\n"
    print("Dcument :",i)

Dcument : 0
Dcument : 1
Dcument : 2
Dcument : 3
Dcument : 4


In [10]:
text

'document no 0 :117. "Microsoft\'s 2018, part 1: Open source, wobbly Windows and everyone\'s going to the cloud"\n(https://www.theregister.co.uk/2018/12/25/microsoft_year_in_review_2018/). The Register.\nArchived (https://web.archive.org/web/20190103060059/https://www.theregister.co.uk/2018/\n12/25/microsoft_year_in_review_2018/) from the original on January 3, 2019. Retrieved\nJanuary 3, 2019.\n\n118. "Microsoft to acquire GitHub for $7.5 billion" (https://news.microsoft.com/2018/06/04/microso\nft-to-acquire-github-for-7-5-billion/). Microsoft. June 4, 2018. Archived (https://web.archive.or\ng/web/20180604142244/https://news.microsoft.com/2018/06/04/microsoft-to-acquire-github-\nfor-7-5-billion/) from the original on June 4, 2018.\ndocument no 1 :In April 2018, Microsoft released the source code for Windows File Manager under the MIT License to\ncelebrate the program\'s 20th anniversary. In April the company further expressed willingness to embrace\nopen source initiatives by announci

In [11]:
#Initialize user query

query = "How much did Microsoft pay to acquire GitHub?"


In [12]:
# Combine the query and the relevant document contents
# Create the multi line combined input ( """ )

combined_input = f"""
You are a helpful assistant. Answer ONLY using the context below.

Context:{text}

Question: {query}

If answer is not in context, say you don't know.
"""

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Create a ChatOpenAI model
model = ChatOpenAI(model="gpt-4o")

In [14]:
# Define the messages for the model
messages = [
    SystemMessage(content = "You are a helpful assistant."),
    HumanMessage(content = combined_input)
]

# Invoke the model with the combined input
answer = model.invoke(messages)

In [ ]:
#Print final output

answer.content

'Microsoft paid $7.5 billion to acquire GitHub.'